In [1]:
import numpy as np
import pandas as pd
from astropy import units as u
from astropy.coordinates import SkyCoord
from astroquery.simbad import Simbad
import matplotlib.pyplot as plt

import requests
import re
from PIL import Image
from io import BytesIO


In [ ]:
# Load the data
df = pd.read_csv("All_Candidates_Summary_b_onder30deg_30per.csv")

max_allowed_radius_pc = 50
df_filtered = df[df['radius_pc'] <= max_allowed_radius_pc].copy()

print(f"Starting SIMBAD analysis for {len(df_filtered)} detections...\n")

# Configure SIMBAD
Simbad.reset_votable_fields()
Simbad.add_votable_fields('otype', 'ra(d)', 'dec(d)', 'plx_value')
Simbad.ROW_LIMIT = 5000
valid_cluster_types = ['OpC', 'GlC', 'ClG', 'GrG', 'As', 'Cl', 'Assoc', 'Cluster']

all_results = []

# Match each detection individually
for index, row in df_filtered.iterrows():
    ra_c = row['ra_center']
    dec_c = row['dec_center']
    dist_kpc = row['distance_kpc']
    
    radius_pc_marge = row['radius_pc'] * 1.2
    afstand_pc = dist_kpc * 1000
    
    zoekhoek_rad = np.arctan(radius_pc_marge / afstand_pc)
    search_radius_deg = np.degrees(zoekhoek_rad)
    search_radius = search_radius_deg * u.deg
    
    coord = SkyCoord(ra=ra_c * u.deg, dec=dec_c * u.deg, frame='icrs', obstime='J2016.0')
    
    try:
        result_table = Simbad.query_region(coord, radius=search_radius)
    except:
        result_table = None
        
    best_match_name = "None"
    best_match_type = "None"
    best_match_dist = "Unknown"
    
    if result_table is not None and len(result_table) > 0:
        best_score = float('inf')
        
        for s_row in result_table:
            otype = str(s_row['otype']).strip().replace('?', '').replace('*', '')
            
            if any(key in otype for key in valid_cluster_types):
                s_coord = SkyCoord(ra=float(s_row['ra']) * u.deg, dec=float(s_row['dec']) * u.deg, frame='icrs')
                sep_deg = coord.separation(s_coord).deg
                
                simbad_dist_kpc = None
                try:
                    s_plx = float(s_row['PLX_value'])
                    if s_plx > 0:
                        # Convert parallax (mas) to distance (kpc)
                        simbad_dist_kpc = 1.0 / s_plx
                except:
                    pass
                
                if sep_deg < best_score:
                    best_score = sep_deg
                    best_match_name = str(s_row['main_id']).strip()
                    best_match_type = str(s_row['otype']).strip()
                    if simbad_dist_kpc is not None:
                        best_match_dist = f"{simbad_dist_kpc:.2f}"

    all_results.append({
        'RA_center': round(ra_c, 4),
        'Dec_center': round(dec_c, 4),
        'Our_Radius_pc': round(row['radius_pc'], 2),
        'Our_Dist_kpc': dist_kpc,
        'Simbad_Name': best_match_name,
        'Simbad_Type': best_match_type,
        'Simbad_Dist_kpc': best_match_dist,
        'Members': row['n_members'],
        'Source_File': row['source_file']
    })

results_df = pd.DataFrame(all_results)

# Merge results based on SIMBAD name
known_raw = results_df[results_df['Simbad_Name'] != "None"]
new_candidates = results_df[results_df['Simbad_Name'] == "None"]

if not known_raw.empty:
    known_clusters = known_raw.groupby('Simbad_Name').agg({
        'Simbad_Type': 'first',
        'Simbad_Dist_kpc': 'first',
        'RA_center': 'mean',
        'Dec_center': 'mean',
        'Our_Dist_kpc': lambda x: ", ".join(map(str, sorted(x.unique()))),
        'Our_Radius_pc': 'mean',
        'Members': lambda x: ", ".join(map(str, x.tolist())),
        'Source_File': lambda x: " | ".join(x.unique())
    }).reset_index()
else:
    known_clusters = known_raw

known_clusters.to_csv("Known_Simbad_Clusters_Matched_b_onder30deg_30per.csv", index=False)
new_candidates.to_csv("Potential_New_Clusters_Matched_b_onder30deg_30per.csv", index=False)

# Display results
print("SIMBAD match results:")
print(f"Total detections analysed      : {len(results_df)}")
print(f"Unique known SIMBAD clusters   : {len(known_clusters)}")
print(f"New potential clusters         : {len(new_candidates)}")

if len(new_candidates) > 0:
    print("Possible new candidates found at the following positions:")
    print(new_candidates[['RA_center', 'Dec_center', 'Our_Dist_kpc', 'Members']].to_string(index=False))
else:
    print("No new candidates found. All clusters are already known in SIMBAD.")

In [ ]:
# Load the data
df_known_n = pd.read_csv("Known_Simbad_Clusters_Matched_b_boven30deg_30per.csv")
df_known_z = pd.read_csv("Known_Simbad_Clusters_Matched_b_onder30deg_30per.csv")
df_new_z = pd.read_csv("Potential_New_Clusters_Matched_b_onder30deg_30per.csv")

# Empty DataFrame for north (no new candidates were found)
df_new_n = pd.DataFrame()

# Print summary
print("=== RESULTS NORTH (b > 30) ===")
print(f"Known SIMBAD clusters: {len(df_known_n)}")
print(f"New potential clusters: {len(df_new_n)}")
print("\n=== RESULTS SOUTH (b < -30) ===")
print(f"Known SIMBAD clusters: {len(df_known_z)}")
print(f"New potential clusters: {len(df_new_z)}")

# Created stack bar chart
# Add hemisphere labels
df_known_n['Hemisphere'] = 'North'
df_known_z['Hemisphere'] = 'South'

# Combine into a single table
df_combined = pd.concat([df_known_n, df_known_z], ignore_index=True)

# Remove asterisks and question marks from SIMBAD types
df_combined['Simbad_Type'] = df_combined['Simbad_Type'].astype(str).str.replace('*', '', regex=False).str.replace('?', '', regex=False)

# Create a crosstab
plot_data = pd.crosstab(df_combined['Simbad_Type'], df_combined['Hemisphere'])

# Force the order of object types to match the valid_cluster_types list
valid_order = ['OpC', 'GlC', 'ClG', 'GrG', 'As', 'Cl', 'Assoc', 'Cluster']
found_types = df_combined['Simbad_Type'].unique().tolist()
ordered_types = [t for t in valid_order if t in found_types] + [t for t in found_types if t not in valid_order]
plot_data = plot_data.reindex(ordered_types)

# Ensure both columns exist, then order North before South
if 'North' not in plot_data.columns: plot_data['North'] = 0
if 'South' not in plot_data.columns: plot_data['South'] = 0
plot_data = plot_data[['North', 'South']]

# Plot (orange for north, blue for south)
plot_data.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#ff7f0e', '#1f77b4'])
plt.xlabel('SIMBAD Object Type')
plt.ylabel('Number of Objects')
plt.title('Identified SIMBAD Object Types per Hemisphere', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Hemisphere')
plt.tight_layout()

# Save and display
plt.savefig('simbad_types_stacked_bar.png', dpi=150)
plt.show()
print(plot_data)

In [ ]:
# Load the file and set the ID
df = pd.read_csv("Potential_New_Clusters_Matched_b_onder30deg_30per.csv")
df['ID'] = df.index

# Select the relevant columns
table_df = df[['ID', 'RA_center', 'Dec_center', 'Our_Dist_kpc', 'Our_Radius_pc', 'Members']].copy()

# Format the numbers directly in the DataFrame
table_df['RA_center'] = table_df['RA_center'].apply(lambda x: f"{x:.4f}")
table_df['Dec_center'] = table_df['Dec_center'].apply(lambda x: f"{x:.4f}")
table_df['Our_Dist_kpc'] = table_df['Our_Dist_kpc'].apply(lambda x: f"{x:.4f}")
table_df['Our_Radius_pc'] = table_df['Our_Radius_pc'].apply(lambda x: f"{x:.4f}")

# Rename columns for display
table_df.columns = ['ID', 'RA (deg)', 'Dec (deg)', 'd (kpc)', 'r (pc)', 'Members (N)']

# Create the figure
fig, ax = plt.subplots(figsize=(8, 16))
ax.axis('off')
ax.axis('tight')

# Draw the table
table = ax.table(cellText=table_df.values,
                 colLabels=table_df.columns,
                 loc='center',
                 cellLoc='center')

# Style the table
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.2)

# Make the header row bold
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(fontweight='bold')

# Save as high resolution PNG
plt.tight_layout()
plt.savefig("candidate_table_image.png", dpi=300, bbox_inches='tight')
print("Table saved as candidate_table_image.png!")
plt.show()

In [ ]:
input_file = "Potential_New_Clusters_Matched_b_onder30deg_30per.csv"

# Load and sort data by RA and Dec
df = pd.read_csv(input_file)
df = df.sort_values(['RA_center', 'Dec_center'])
print(f"Total number of candidates: {len(df)}")

# Check whether position falls within PanSTARRS coverage
def within_panstarrs_coverage(dec):
    # PanSTARRS covers the sky above Dec > -30 degrees
    return dec > -30

# Fetch PanSTARRS image
def get_panstarrs_image(ra, dec, radius_deg, size_px=600):
    # Determine field of view: 2.5x the cluster radius with a minimum and maximum
    fov_arcsec = radius_deg * 3600 * 2.5
    fov_arcsec = max(fov_arcsec, 60)    # minimum 60 arcsec
    fov_arcsec = min(fov_arcsec, 7200)  # maximum 2 degrees
    size_pixels = int(fov_arcsec / 0.25)

    # Retrieve available FITS files for this position
    info_url = (
        f"https://ps1images.stsci.edu/cgi-bin/ps1cutouts?"
        f"pos={ra}+{dec}&filter=color&filetypes=stack"
        f"&size={size_pixels}&output_size={size_px}"
    )
    try:
        response = requests.get(info_url, timeout=30)

        # Find FITS file links for g, r and i filters
        fits_links = re.findall(r'href="(/rings[^"]+\.fits)"', response.text)
        g_fits = next((l for l in fits_links if '.stk.g.' in l), None)
        r_fits = next((l for l in fits_links if '.stk.r.' in l), None)
        i_fits = next((l for l in fits_links if '.stk.i.' in l), None)

        # If not all filters are available, return None
        if not all([g_fits, r_fits, i_fits]):
            return None

        # Build the colour image URL via the fitscut API (i=red, r=green, g=blue)
        color_url = (
            f"https://ps1images.stsci.edu/cgi-bin/fitscut.cgi?"
            f"red={i_fits}&green={r_fits}&blue={g_fits}"
            f"&format=jpg&x={ra}&y={dec}&size={size_pixels}"
            f"&wcs=1&autoscale=99.5&output_size={size_px}"
        )
        img_response = requests.get(color_url, timeout=60)
        if img_response.status_code == 200:
            return Image.open(BytesIO(img_response.content))
        else:
            return None

    except Exception as e:
        print(f"Error fetching image for RA={ra}, Dec={dec}: {e}")
        return None

# Main loop: generate one figure per candidate
for idx, row in df.iterrows():

    # Retrieve basic properties
    ra      = row['RA_center']
    dec     = row['Dec_center']
    dist    = float(row['Our_Dist_kpc'])
    members = int(row['Members'])

    # Calculate the angular size of the cluster in degrees
    radius_deg = row['Our_Radius_pc'] / (dist * 1000) * (180/np.pi)

    # Skip candidates outside PanSTARRS coverage
    if not within_panstarrs_coverage(dec):
        print(f"Candidate {row.name}: Dec={dec:.2f} outside PanSTARRS coverage, skipped.")
        continue

    # Fetch the PanSTARRS colour image
    img = get_panstarrs_image(ra, dec, radius_deg)

    if img is not None:

        # Print the URL so the image can be downloaded directly
        fov_arcsec = radius_deg * 3600 * 2.5
        fov_arcsec = max(fov_arcsec, 60)
        fov_arcsec = min(fov_arcsec, 7200)
        size_pixels = int(fov_arcsec / 0.25)
        print(f"Candidate #{row.name} — RA: {ra:.4f}, Dec: {dec:.4f}")
        print(f"PanSTARRS URL: https://ps1images.stsci.edu/cgi-bin/ps1cutouts?pos={ra}+{dec}&filter=color&filetypes=stack&size={size_pixels}&output_size=600")
        print()

        img_array = np.array(img)
        h, w = img_array.shape[:2]

        # Create figure sized to match the image, with extra space for text
        dpi = 100
        fig_w = w / dpi
        fig_h = h / dpi + 1.2  # 1.2 extra for title and subtitle

        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        fig.subplots_adjust(top=0.88, bottom=0.12, left=0, right=1)

        ax.imshow(img_array)
        ax.axis('off')
    else:
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.set_facecolor('black')
        ax.text(0.5, 0.5, 'No PanSTARRS data available',
                ha='center', va='center', color='white', fontsize=12,
                transform=ax.transAxes)
        ax.axis('off')

    # Title with candidate number and position
    ax.set_title(f"Candidate #{row.name}  —  RA: {ra:.4f}°  |  Dec: {dec:.4f}°",
                fontsize=13, fontweight='bold', pad=8)

    # Subtitle with remaining cluster properties
    subtitle = (
        f"Distance: {dist:.3f} kpc  |  Radius: {row['Our_Radius_pc']:.2f} pc  "
        f"|  Angular size: {radius_deg*60:.2f} arcmin  |  Members: {members}"
    )
    fig.text(0.5, 0.02, subtitle, ha='center', fontsize=10, color='#333333')

    # Save the figure as PNG
    plt.savefig(f"panstarrs_candidate_{row.name:03d}_ra{ra:.2f}_dec{dec:.2f}.png",
                dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close()

print(f"\nDone")

In [ ]:
input_file = "Potential_New_Clusters_Matched_b_onder30deg_30per.csv"

# Load and sort data by RA and Dec
df = pd.read_csv(input_file)
df = df.sort_values(['RA_center', 'Dec_center'])
print(f"Total number of candidates: {len(df)}")

# Check whether position falls within SkyMapper coverage
def within_skymapper_coverage(dec):
    # SkyMapper covers the sky below Dec < +16 degrees (DR4)
    return dec < 16

def get_skymapper_image(ra, dec, radius_deg):
    # Determine field of view: 2.5x the cluster radius with a minimum and maximum
    fov_deg = radius_deg * 2.5
    fov_deg = max(fov_deg, 60/3600)    # minimum 60 arcsec
    fov_deg = min(fov_deg, 600/3600)   # maximum 600 arcsec (SkyMapper limit)

    query_url = (
        f"https://api.skymapper.nci.org.au/public/siap/dr4/query?"
        f"POS={ra},{dec}&SIZE={fov_deg}&BAND=g,r,i"
        f"&FORMAT=image/png&INTERSECT=CENTER&VERB=3"
    )

    try:
        response = requests.get(query_url, timeout=30)
        if response.status_code != 200:
            return None

        # Parse the VOTable response
        votable = parse(BytesIO(response.content), verify='ignore')
        table = votable.get_first_table().to_table()

        if len(table) == 0:
            return None

        # Find the URLs per filter
        g_url = r_url = i_url = None
        for row in table:
            band = str(row['band']).strip()
            url  = str(row['get_image']).strip()
            if band == 'g' and g_url is None:
                g_url = url
            elif band == 'r' and r_url is None:
                r_url = url
            elif band == 'i' and i_url is None:
                i_url = url

        # If not all filters are available, return None
        if not all([g_url, r_url, i_url]):
            return None

        # Download the three filter images
        def download_png(url):
            r = requests.get(url, timeout=60)
            if r.status_code == 200:
                img = Image.open(BytesIO(r.content)).convert('L')
                return np.array(img, dtype=float)
            return None

In [ ]:
# Settings
candidates_file = "Potential_New_Clusters_Matched_b_onder30deg_30per.csv"
results_dir = "Results_b_onder30deg_30per"
candidate_ids = [35, 42, 63]  # Adjust these IDs to match the candidates discussed in the thesis
circle_radius = 10  # Size of the circles in arcseconds (change this number to make them larger/smaller)

# Load the candidate list
df = pd.read_csv(candidates_file)

for cand_id in candidate_ids:
    row = df.loc[cand_id]
    source_file = f"{results_dir}/{row['Source_File']}"
    stars = pd.read_csv(source_file)
    
    print(f"Candidate {cand_id}: {len(stars)} stars found")
    
    # Create the region file
    reg_filename = f"candidate_{cand_id:03d}.reg"
    
    with open(reg_filename, 'w') as f:
        # Header
        f.write("# Region file format: DS9 version 4.1\n")
        # global width=2 makes the lines slightly thicker
        f.write("global color=green dashlist=8 3 width=2 font=\"helvetica 10 normal roman\"\n")
        f.write("fk5\n")
        
        # Circle per star without label
        for _, star in stars.iterrows():
            f.write(f"circle({star['ra']:.6f},{star['dec']:.6f},{circle_radius}\")\n")
    
    print(f"Region file saved as: {reg_filename}\n")

print("Done!")